# Extracting EDINET Control Variables
This notebook outlines the process of extracting key financial metrics ("control variables") from EDINET CSV filings and loading them into a relational database for analysis. We focus on annual securities reports (有価証券報告書) and retrieve metrics such as total assets, liabilities, net assets, revenue, operating income, and net income for each company and fiscal year. The data will be pivoted to a wide format (one row per company-year) and upserted into a PostgreSQL table (`edinet_controls`).

## Configuration and Database Connection
First, we set up the environment and database connection. We allow flexible specification of the fiscal year range to process (defaulting to 2018–2024). We also ensure the working directory is set to the project root so that relative file paths (for CSV data) are correctly resolved. Database credentials are loaded from environment variables (e.g., using a .env file).

In [1]:
import os
import sys
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Ensure working directory is project root (if this notebook is in a subfolder)
os.chdir(os.path.abspath(".."))
sys.path.append(os.path.abspath(".."))

# Load database connection info from environment
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Define the fiscal year range for processing
YEAR_START = 2017
YEAR_END   = 2025

print(f"Processing fiscal years {YEAR_START} to {YEAR_END}")

Processing fiscal years 2017 to 2025


## Retrieve Target Documents from EDINET
We query the `edinet_documents` table to get metadata for all annual securities reports in the specified year range. We filter by `doc_type_code` corresponding to annual reports (in EDINET, code "120" or "120000" denotes annual securities reports). 
The query will retrieve each document's unique ID, EDINET code (company identifier), company_id, fiscal year, submission date, and the local CSV file path. We also ensure that a CSV path is available (CSV files were downloaded via the EDINET API for these documents).

In [2]:
# Query the edinet_documents table for annual report documents in the given year range
query = """
SELECT company_id, edinet_code, doc_id, doc_type_code, fiscal_year, submit_date, local_csv_path
FROM edinet_documents
WHERE doc_type_code IN ('120')
  AND fiscal_year BETWEEN %(start_year)s AND %(end_year)s
  AND local_csv_path IS NOT NULL
ORDER BY fiscal_year, edinet_code;
"""
documents_df = pd.read_sql(query, engine, params={"start_year": YEAR_START, "end_year": YEAR_END})

print(f"Retrieved {len(documents_df)} documents for annual reports from {YEAR_START} to {YEAR_END}.")
documents_df.head(5)

Retrieved 8647 documents for annual reports from 2017 to 2025.


,company_id,edinet_code,doc_id,doc_type_code,fiscal_year,submit_date,local_csv_path
0,8bfff503-65a2-4572-a8c3-0f44a1fc9fe3,E00011,S100AH2F,120,2017,2017-06-23 12:01:00,data/raw/edinet_csv_data/2017/E00011/S100AH2F.csv
1,5687659d-c269-4057-b7a3-a494c305e7a6,E00014,S100AJIN,120,2017,2017-06-28 15:57:00,data/raw/edinet_csv_data/2017/E00014/S100AJIN.csv
2,42fd0f65-38c4-4d7c-a809-36c1dbdbafa9,E00015,S100AIUC,120,2017,2017-06-28 13:52:00,data/raw/edinet_csv_data/2017/E00015/S100AIUC.csv
3,f952837e-12cc-4af9-b64d-058d9b3cf75a,E00021,S100AJT3,120,2017,2017-06-28 13:38:00,data/raw/edinet_csv_data/2017/E00021/S100AJT3.csv
4,8ca4ba0e-8a05-4ed3-a8d5-1b1544027739,E00023,S100AOAB,120,2017,2017-06-27 16:18:00,data/raw/edinet_csv_data/2017/E00023/S100AOAB.csv


_Explanation_: We expect one document per company per fiscal year (in most cases). Each entry provides the path to a CSV file stored under `data/raw/edinet_csv_data/<year>/<edinet_code>/<doc_id>.csv`, which contains the XBRL data converted to CSV for that report.

## Define Metric Mapping (raw2metric)
EDINET's CSV files list many data points identified by 要素ID (element IDs) and other context columns. We prepare a dictionary `raw2metric` to map these EDINET element IDs to our desired metric names. This mapping helps filter and rename the relevant financial figures. In cases where multiple EDINET elements could correspond to the same metric, we include all possibilities and will choose the most appropriate (typically consolidated values over unconsolidated, and current-year figures over prior-year figures). For example, the element ID `jpcrp_cor:TotalAssetsSummaryOfBusinessResults` (used in the "Key Financial Data" section of the report) represents "Total Assets"
. We map this to `total_assets`. We do similarly for liabilities, net assets, revenue, operating income, and net income. We include both consolidated (連結) and unconsolidated (個別) element variants, as well as Japanese GAAP vs. IFRS taxonomy variants (some IFRS-based element names include "JMIS" in the ID).

In [3]:
# Mapping of raw EDINET element IDs (要素ID) to standardized metric names
raw2metric = {
    # Total Assets
    "jppfs_cor:Assets": "total_assets",  # Consolidated total assets (IFRS actual BS)
    # Total Liabilities
    "jppfs_cor:Liabilities": "total_liabilities",  # Consolidated total liabilities (IFRS)
    # Net Assets (Total Equity)
    "jppfs_cor:NetAssets": "net_assets",
    # Revenue (Net Sales)
    "jppfs_cor:NetSales": "revenue",
    "jpcrp_cor:NetSalesSummaryOfBusinessResults": "revenue",
    "jppfs_cor:OperatingRevenue1": "revenue",
    # Operating Income (Operating Profit)
    "jppfs_cor:OperatingIncome": "operating_income",  # Operating income (IFRS, if reported)
    # Net Income (Profit attributable to owners of parent)
    "jppfs_cor:ProfitLossAttributableToOwnersOfParent": "net_income",
    "jppfs_cor:ProfitLoss": "net_income", # Net income attributable to owners (IFRS consolidated)
}
print(f"Defined raw2metric mappings for {len(raw2metric)} element IDs.")

Defined raw2metric mappings for 9 element IDs.


_Note_: The element IDs are case-sensitive and include taxonomy prefixes (e.g., jpcrp_cor for common reporting taxonomy, jppfs_cor for public financial statement taxonomy under JMIS/IFRS). By including multiple IDs per metric, we cover variations in reports. When extracting, if multiple IDs for the same metric are present, we will prioritize consolidated values. (The dictionary is ordered; we insert consolidated/primary concepts first, so once a metric is captured we won't override it with secondary entries.)
## Parse CSV Files and Extract Metrics
Next, we iterate over each document, read its CSV file, and extract the relevant metrics. The EDINET CSV files are encoded in UTF-16 and use tab (`\t`) as a delimiter
, so we read them accordingly. Each CSV contains rows of data identified by `要素ID` and columns indicating context (e.g., relative year, consolidated vs. individual). We will filter each CSV to the rows that represent the current year (相対年度=0) and consolidated data where available. If consolidated data is not available (i.e. the company only has non-consolidated statements), we will use the non-consolidated values.


For each such filtered row, if the 要素ID is in our raw2metric mapping, we record its value. We ensure that if multiple rows map to the same metric, we take the first (which, by our mapping order, should be the consolidated figure).

In [4]:
# === Cell: Extract metrics from EDINET CSVs (fixed) ===
import os
import numpy as np
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

records = []
first = True

for _, row in documents_df.iterrows():
    doc_id      = row["doc_id"]
    edinet_code = row["edinet_code"]
    company_id  = row["company_id"]
    fiscal_year = row["fiscal_year"]
    csv_path    = row["local_csv_path"]
    
    # 1) Load CSV
    try:
        df = pd.read_csv(csv_path, encoding="utf-16", sep="\t")
    except Exception as e:
        logger.warning(f"⚠️ Could not read CSV for {doc_id}: {e}")
        continue
    
    # 2) Normalize
    df.columns  = df.columns.str.strip()
    df["要素ID"] = df["要素ID"].astype(str).str.strip()
    
    # 3) Keep only current year (当期 or 当期末)
    if "相対年度" in df.columns:
        if np.issubdtype(df["相対年度"].dtype, np.number):
            df = df[df["相対年度"] == 0]
        else:
            df = df[df["相対年度"].isin(["当期", "当期末"])]
    
    # Prepare container
    metrics_values = {
        "company_id":  company_id,
        "edinet_code": edinet_code,
        "fiscal_year": fiscal_year
    }
    
    found, missing = [], []
    # Pull in each mapping
    for element_id, metric in raw2metric.items():
        hits = df[df["要素ID"] == element_id]
        if hits.empty:
            missing.append(element_id)
            continue
        
        # 4) Consolidated vs non-consolidated filter
        is_consolidated = False
        if "連結・個別" in hits.columns:
            is_consolidated = hits["連結・個別"].eq("連結").any()
            if is_consolidated:
                hits = hits[hits["連結・個別"] == "連結"]
            else:
                hits = hits[hits["連結・個別"].isin(["個別", "その他"])]
            if hits.empty:
                continue
        
        # 5) Context-ID selection
        if metric in ("total_assets", "total_liabilities", "net_assets", "revenue"):
            # balance sheet → Instant contexts
            contexts = ["CurrentYearInstant"]
            if not is_consolidated:
                contexts.append("CurrentYearInstant_NonConsolidatedMember")
        else:
            # P/L → Duration contexts
            contexts = ["CurrentYearDuration"]
            if not is_consolidated:
                contexts.append("CurrentYearDuration_NonConsolidatedMember")
        
        for ctx in contexts:
            if "コンテキストID" in hits.columns:
                subset = hits[hits["コンテキストID"] == ctx]
                if not subset.empty:
                    hits = subset
                    break
        else:
            # no matching context
            continue
        
        # 6) Extract and convert raw value
        raw = hits.iloc[0]["値"]
        if pd.isna(raw):
            missing.append(element_id)
            continue
        if isinstance(raw, np.generic):
            raw = raw.item()
        if isinstance(raw, str):
            s = raw.replace(",", "")
            if s.replace(".", "", 1).lstrip("+-").isdigit():
                raw = int(s) if s.isdigit() else float(s)
        
        metrics_values[metric] = raw
        found.append(element_id)
    
    # debug first iteration
    if first:
        logger.info(f"--- DEBUG for {doc_id} ---\n  Found IDs:    {found}\n  Missing IDs: {missing[:8]}…")
        first = False
    
    # 7) Keep only if we got at least one metric
    if len(metrics_values) > 3:
        records.append(metrics_values)

logger.info(f"Extracted metrics for {len(records)} company-year documents.")

INFO:__main__:--- DEBUG for S100AH2F ---
  Found IDs:    ['jppfs_cor:Assets', 'jppfs_cor:Liabilities', 'jppfs_cor:NetAssets', 'jppfs_cor:NetSales', 'jpcrp_cor:NetSalesSummaryOfBusinessResults', 'jppfs_cor:OperatingIncome', 'jppfs_cor:ProfitLossAttributableToOwnersOfParent', 'jppfs_cor:ProfitLoss']
  Missing IDs: ['jppfs_cor:OperatingRevenue1']…
INFO:__main__:Extracted metrics for 8643 company-year documents.


In this loop:

- We use `pd.read_csv` with the proper encoding and separator

- We filter by `相対年度 == 0` to focus on the current reporting year (ignoring any prior-year comparative figures that might be present).
- We then filter to **連結 (consolidated)** if available. The CSV has a column "連結・個別" indicating whether the data is consolidated or individual
- We prefer consolidated data for our control variables. If a company has only individual data (no subsidiaries), we will use that.
- For each element ID in our mapping, we find if it exists in the filtered dataframe. We then take its "値" (value). We ensure that if a metric is already found (e.g., if both a consolidated and an unconsolidated entry for essentially the same metric existed, or a summary and a detailed entry), we don't overwrite it. The ordering of keys in `raw2metric` ensures the preferred source is handled first.
- We perform some type conversions: numeric values are left as numbers; strings that contain numbers are converted to int/float; missing values are skipped.
- We accumulate each company's metrics in the `records` list as a dictionary.

## Prepare Wide-Format DataFrame
Now we convert the collected records into a pandas DataFrame. This will naturally be in a wide format: each row corresponds to a unique combination of `company_id` (and `edinet_code`) and `fiscal_year`, and each metric is in a separate column. We set `edinet_code` and `fiscal_year` as a composite index for clarity (optional). We'll also replace any missing values with `None` (so they will be stored as SQL NULL in the database).

In [5]:
# Create DataFrame from the records list
controls_df = pd.DataFrame(records)

# Set multi-index (edinet_code, fiscal_year) for easier viewing (optional)
controls_df.set_index(["edinet_code", "fiscal_year"], inplace=True)

# Replace NaN with None for database compatibility
controls_df = controls_df.where(pd.notnull(controls_df), None)

print(f"Final dataframe has {controls_df.shape[0]} rows and {controls_df.shape[1]} columns (including ID columns).")
controls_df.head(10)

Final dataframe has 8643 rows and 7 columns (including ID columns).


,,company_id,total_assets,total_liabilities,net_assets,revenue,operating_income,net_income
edinet_code,fiscal_year,,,,,,,
E00011,2017,8bfff503-65a2-4572-a8c3-0f44a1fc9fe3,793617000000,498273000000,2.953440e+11,1113364000000,53989000000,4.023000e+10
E00014,2017,5687659d-c269-4057-b7a3-a494c305e7a6,451876000000,310671000000,1.412050e+11,635953000000,22646000000,1.559600e+10
E00015,2017,42fd0f65-38c4-4d7c-a809-36c1dbdbafa9,501303000000,378482000000,1.228200e+11,873295000000,26308000000,1.881400e+10
E00021,2017,f952837e-12cc-4af9-b64d-058d9b3cf75a,1896939000000,1186744000000,7.101950e+11,1304068000000,59761000000,3.557300e+10
E00023,2017,8ca4ba0e-8a05-4ed3-a8d5-1b1544027739,1685018000000,660897000000,1.024121e+12,786146000000,76390000000,-2.904500e+10
E00024,2017,7fdfe904-7ea1-4c05-b8a6-5978ce9fd5f8,518981000000,334560000000,1.844210e+11,436330000000,38461000000,1.960500e+10
E00028,2017,863b6db0-6e13-49a9-8eb7-66335484dbe9,404604000000,176782000000,2.278210e+11,410503000000,33990000000,2.647400e+10
E00043,2017,86b04b3f-70df-4203-95ff-4e8bc04e8322,4312174000000,1104631000000,3.207542e+12,874423000000,336452000000,5.613100e+10
E00048,2017,616bdad1-86ee-4fcd-855a-4173101636eb,3555885000000,2225983000000,1.329901e+12,3512909000000,310092000000,2.027920e+11


The DataFrame now contains our control variables in wide format, with columns: `company_id`, and each of the six metrics (`total_assets`, `total_liabilities`, `net_assets`, `revenue`, `operating_income`, `net_income`). The index is set to (`edinet_code`, `fiscal_year`) for uniqueness, though the `company_id` is also present as an identifier. (If needed, one could also pivot from a long format, but here we built the wide format directly.)

## Upsert into PostgreSQL (`edinet_controls` table)
Finally, we create or update the target table in the PostgreSQL database. The table `edinet_controls` will store the control variables for each company and fiscal year. We define the schema with appropriate data types (here we use numeric for financial values, but these could also be BigInt if we expect only integer yen amounts). The primary key will be a composite of `company_id` and `fiscal_year`, ensuring one record per company-year. 

We then perform an upsert (insert or update) for each record. If a record with the same `company_id` and `fiscal_year` already exists, we update the values; if not, we insert a new row. This ensures the operation is idempotent and can be rerun without duplicating data.

In [6]:
import numpy as np
import pandas as pd

# List of columns to clean & convert
numeric_cols = [
    "total_assets", "total_liabilities", "net_assets",
    "revenue", "operating_income", "net_income"
]

# 1) 全角ダッシュ を None に置き換え
controls_df[numeric_cols] = controls_df[numeric_cols].replace("－", None)

# 2) 数値型に変換 (変換できない文字列は NaN になる)
controls_df[numeric_cols] = controls_df[numeric_cols].apply(
    lambda col: pd.to_numeric(col, errors="coerce")
)

# === Cell Y: Create table & bulk upsert controls ===
from sqlalchemy import text

# Define table creation SQL (if the table might not exist yet)
create_table_sql = text("""
CREATE TABLE IF NOT EXISTS edinet_controls (
    company_id    UUID            NOT NULL,
    fiscal_year   INTEGER         NOT NULL,
    total_assets         NUMERIC,
    total_liabilities    NUMERIC,
    net_assets           NUMERIC,
    revenue              NUMERIC,
    operating_income     NUMERIC,
    net_income           NUMERIC,
    PRIMARY KEY (company_id, fiscal_year)
);
""")
# Execute table creation
with engine.begin() as conn:
    conn.execute(create_table_sql)

# Prepare upsert (INSERT ... ON CONFLICT) statement
upsert_sql = text("""
INSERT INTO edinet_controls 
    (company_id, fiscal_year, total_assets, total_liabilities, net_assets, revenue, operating_income, net_income)
VALUES 
    (:company_id, :fiscal_year, :total_assets, :total_liabilities, :net_assets, :revenue, :operating_income, :net_income)
ON CONFLICT (company_id, fiscal_year)
DO UPDATE SET 
    total_assets      = EXCLUDED.total_assets,
    total_liabilities = EXCLUDED.total_liabilities,
    net_assets        = EXCLUDED.net_assets,
    revenue           = EXCLUDED.revenue,
    operating_income  = EXCLUDED.operating_income,
    net_income        = EXCLUDED.net_income;
""")

# Convert DataFrame to list of dicts for bulk upsert
records = controls_df.reset_index()[[
    "company_id", "fiscal_year", "total_assets", "total_liabilities",
    "net_assets", "revenue", "operating_income", "net_income"
]].to_dict(orient="records")

# Perform upsert in a single batch
with engine.begin() as conn:
    conn.execute(upsert_sql, records)

print(f"Upserted {len(records)} records into edinet_controls table.")

Upserted 8643 records into edinet_controls table.


We use a parameterized SQL statement with SQLAlchemy's `text`. Each row from our DataFrame is inserted with the defined columns. The `ON CONFLICT` clause handles duplicates by updating the existing record with any new values (this covers cases where we re-run for the same data or if new data for the same key comes in). After running the above, the `edinet_controls` table will be populated (or updated) with the control variables for each company and fiscal year in our range. Each record contains:
- `company_id` – a reference to the company (linked to a companies master table if available),
- `fiscal_year` – the year of the report,
- `total_assets`, `total_liabilities`, `net_assets` – balance sheet figures (typically as of fiscal year end),
- `revenue`, `operating_income`, `net_income` – performance figures for that fiscal year.

This completes the extraction and loading process. The data is now ready to be used in analysis or modeling as needed.

In [7]:
company_id = "16f8c0f3-596a-4f53-8396-1dc1bd5e7d9a"

# cast the UUID column to str before comparing
mask = documents_df["company_id"].astype(str) == company_id
docs_for_company = documents_df.loc[mask, 
    ["doc_id", "fiscal_year", "doc_type_code", "local_csv_path"]]

print(docs_for_company)

        doc_id  fiscal_year doc_type_code  \
100   S100AJM3         2017           120   
864   S100D9Z7         2018           120   
1665  S100G64E         2019           120   
2509  S100ISZU         2020           120   
3411  S100LNZQ         2021           120   
4391  S100OCR3         2022           120   
5453  S100R509         2023           120   
6582  S100TQ91         2024           120   
7767  S100W2HS         2025           120   

                                         local_csv_path  
100   data/raw/edinet_csv_data/2017/E00877/S100AJM3.csv  
864   data/raw/edinet_csv_data/2018/E00877/S100D9Z7.csv  
1665  data/raw/edinet_csv_data/2019/E00877/S100G64E.csv  
2509  data/raw/edinet_csv_data/2020/E00877/S100ISZU.csv  
3411  data/raw/edinet_csv_data/2021/E00877/S100LNZQ.csv  
4391  data/raw/edinet_csv_data/2022/E00877/S100OCR3.csv  
5453  data/raw/edinet_csv_data/2023/E00877/S100R509.csv  
6582  data/raw/edinet_csv_data/2024/E00877/S100TQ91.csv  
7767  data/raw/edinet_csv_d